# 01 — Pulizia dei dati MBFC (tentativo 10)

L'esportazione dall'API MBFC (`Dataset/mbfc_api_2026-10-23.json`, 11.176 schede) ha un campo per ogni voce della
scheda: nome, link alla scheda, categoria di bias, paese, factual reporting, tipo di media, sito, credibilità, ID,
factual score, orientamento politico, più una colonna senza nome. Qui si mette in ordine, senza buttare nulla:

1. **sito → dominio**: minuscole, via `http(s)://`, `www.` e la `/` finale; se resta un percorso
   (`frontiersin.org/journals/...`) il dominio è la parte prima della `/` e il percorso si tiene a parte: quella scheda
   valuta una sezione, non il sito;
2. **siti morti**: `DEAD`, `Dead`, `Offline` → `morto` = vero, nessun dominio;
3. **credibilità**: il refuso `Medum` → `Medium`; valori che non sono High/Medium/Low (`N/A` della satira, un `Mixed`,
   un `DEAD`) → vuoti;
4. **factual reporting**: `N/A` (satira) → vuoto;
5. **factual score** (il numero di MBFC: 0 = Very High, 10 = Very Low): `#N/A` e `N/A` → vuoti; una data di Excel
   (`1900-01-05T07:24`) è il numero 5,3 letto come data, e si riconverte; si segnalano i punteggi fuori dalla fascia
   del loro factual reporting;
6. la **colonna senza nome** ha 10 note sparse (proprietario, ...): si tiene come `nota`.

Uscite: `Dataset/mbfc_pulito.csv` e, in `Results/1-etichette`, i passi (`01_passi.csv`) e le distribuzioni
(`01_distribuzioni.csv`).

In [1]:
import json
import re

import numpy as np
import pandas as pd

from comune import CATEGORIA, CREDIBILITA, DATASET, FACTUAL, MBFC_API, ORIENTAMENTO, RESULTS

RESULTS.mkdir(parents=True, exist_ok=True)
grezzo = pd.DataFrame(json.load(open(MBFC_API)))
print(len(grezzo), "schede;", list(grezzo.columns))
m = grezzo.rename(columns={"Source": "nome", "MBFC URL": "url_mbfc", "Bias": "categoria", "Country": "paese",
                           "Factual Reporting": "factual", "Media Type": "tipo_media", "Source URL": "sito",
                           "Credibility": "credibilita", "Source ID#": "id_mbfc", "Factual Score": "factual_score",
                           "Political Bias": "orientamento", "": "nota"})
passi = []
def passo(nome, n):
    passi.append({"passo": nome, "schede": int(n)}); print(f"{nome}: {int(n)}")

11176 schede; ['Source', 'MBFC URL', 'Bias', 'Country', 'Factual Reporting', 'Media Type', 'Source URL', 'Credibility', 'Source ID#', 'Factual Score', 'Political Bias', '']


## 1–2. Dal sito al dominio; i siti morti

In [2]:
s = m.sito.astype(str).str.strip().str.lower()
s = s.str.replace(r"^https?://", "", regex=True).str.replace(r"^www\.", "", regex=True).str.rstrip("/")
m["morto"] = s.isin(["dead", "offline", "", "nan", "none"])
m["dominio"] = s.str.split("/", n=1).str[0].where(~m.morto)
m["percorso"] = s.str.split("/", n=1).str[1].where(~m.morto)
passo("siti morti (DEAD/Offline)", m.morto.sum())
passo("schede di una sezione (dominio + percorso)", m.percorso.notna().sum())
passo("domini diversi (schede vive, senza percorso)", m.dominio[m.percorso.isna()].nunique())
dup = m[m.dominio.notna() & m.percorso.isna()].groupby("dominio").size()
passo("domini con più schede (vive, senza percorso)", (dup > 1).sum())

siti morti (DEAD/Offline): 1091
schede di una sezione (dominio + percorso): 935
domini diversi (schede vive, senza percorso): 9006
domini con più schede (vive, senza percorso): 98


## 3–5. Credibilità, factual reporting, factual score

In [3]:
passo("credibilità 'Medum' corretta in 'Medium'", (m.credibilita == "Medum").sum())
m["credibilita"] = m.credibilita.replace({"Medum": "Medium"})
fuori = ~m.credibilita.isin(CREDIBILITA)
passo("credibilità non High/Medium/Low → vuota (" + ", ".join(f"{k} {v}" for k, v in m.credibilita[fuori].value_counts().items()) + ")", fuori.sum())
m.loc[fuori, "credibilita"] = np.nan
fuori = ~m.factual.isin(FACTUAL)
passo("factual reporting non valido → vuoto (" + ", ".join(f"{k} {v}" for k, v in m.factual[fuori].value_counts().items()) + ")", fuori.sum())
m.loc[fuori, "factual"] = np.nan
for c, ammessi in [("categoria", CATEGORIA), ("orientamento", ORIENTAMENTO)]:
    assert m[c].isin(ammessi).all(), m[c][~m[c].isin(ammessi)].value_counts()

fs = m.factual_score.astype(str)
data_excel = fs.str.match(r"^1900-01-\d\dT")
def da_excel(x):        # numero salvato come data da Excel: 1900-01-01 = 1
    t = pd.Timestamp(x).tz_localize(None)
    return round((t - pd.Timestamp("1899-12-31")).total_seconds() / 86400, 1)
m["factual_score"] = pd.to_numeric(m.factual_score.where(~data_excel), errors="coerce")
m.loc[data_excel, "factual_score"] = fs[data_excel].map(da_excel)
passo("factual score letto come data da Excel → numero (" + ", ".join(fs[data_excel]) + f" → {m.factual_score[data_excel].tolist()})", data_excel.sum())
passo("factual score vuoto (#N/A, N/A)", m.factual_score.isna().sum())
# fasce del factual score per ogni livello (dalla distribuzione: 0 Very High, ~1 High, 2,5-4 Mostly Factual,
# 5-6,1 Mixed, 8 Low, 9,8-10 Very Low); si segnalano i punteggi fuori fascia, senza toccarli
FASCE = {"Very High": (0, 0.5), "High": (0.5, 2.5), "Mostly Factual": (1.5, 5), "Mixed": (4, 7.5), "Low": (5, 9.5), "Very Low": (9, 10)}
lo = m.factual.map(lambda f: FASCE.get(f, (np.nan,))[0]); hi = m.factual.map(lambda f: FASCE.get(f, (np.nan, np.nan))[1])
m["score_incoerente"] = m.factual_score.notna() & m.factual.notna() & ~m.factual_score.between(lo, hi)
passo("factual score fuori dalla fascia del factual reporting (segnalati, non toccati)", m.score_incoerente.sum())
print(m[m.score_incoerente][["nome", "dominio", "factual", "factual_score"]].to_string())

credibilità 'Medum' corretta in 'Medium': 2
credibilità non High/Medium/Low → vuota (N/A 172, DEAD 1, Mixed 1): 174
factual reporting non valido → vuoto (N/A 172): 172
factual score letto come data da Excel → numero (1900-01-05T07:24:00.000Z → [5.3]): 1
factual score vuoto (#N/A, N/A): 200
factual score fuori dalla fascia del factual reporting (segnalati, non toccati): 6
                                   nome             dominio factual  factual_score
336                   Anti-Fascist News                 NaN    High            5.0
343                           Arab News        arabnews.com    High            6.1
439                      Beijing Review        bjreview.com    High            6.1
655                       Brightnews.ai       brightnews.ai     Low            9.8
11002                             EBSCO           ebsco.com    High            0.4
11017  Washington University Law Review  washulawreview.org    High            0.4


## 6. La nota; salvataggio e distribuzioni

In [4]:
m["nota"] = m.nota.replace("", np.nan)
passo("schede con una nota nella colonna senza nome", m.nota.notna().sum())
colonne = ["id_mbfc", "nome", "dominio", "percorso", "morto", "url_mbfc", "categoria", "orientamento", "factual", "factual_score",
           "score_incoerente", "credibilita", "paese", "tipo_media", "nota", "sito"]
m[colonne].to_csv(DATASET / "mbfc_pulito.csv", index=False)
pd.DataFrame(passi).to_csv(RESULTS / "01_passi.csv", index=False)
vive = m[~m.morto]
righe = []
for c, ordine in [("factual", FACTUAL), ("credibilita", CREDIBILITA), ("orientamento", ORIENTAMENTO), ("categoria", CATEGORIA),
                  ("tipo_media", None), ("paese", None)]:
    v = vive[c].value_counts(dropna=False)
    v = v.reindex(ordine + [np.nan]).dropna() if ordine else v.head(15)
    righe += [{"campo": c, "valore": k if isinstance(k, str) else "(vuoto)", "schede": int(n), "quota": n / len(vive)} for k, n in v.items()]
dist = pd.DataFrame(righe); dist.to_csv(RESULTS / "01_distribuzioni.csv", index=False)
print(f"{len(m)} schede, {len(vive)} vive, salvate in Dataset/mbfc_pulito.csv")
dist.assign(quota=dist.quota.round(3))

schede con una nota nella colonna senza nome: 10


11176 schede, 10085 vive, salvate in Dataset/mbfc_pulito.csv


,campo,valore,schede,quota
0,factual,Very High,125,0.012
1,factual,High,5303,0.526
2,factual,Mostly Factual,1203,0.119
3,factual,Mixed,2470,0.245
4,factual,Low,486,0.048
5,factual,Very Low,424,0.042
6,factual,(vuoto),74,0.007
7,credibilita,High,6069,0.602
8,credibilita,Medium,1383,0.137
9,credibilita,Low,2558,0.254
